# Step 2 — Prompt Engineering for Conversation Intelligence
**Tough Talks · Phase 1**

> **Prerequisite:** Step 1 must have run in this session — `model` and `tokenizer` must be loaded.

Goal: Engineer and validate the 4 core prompts that drive all Tough Talks features:
1. **Adversarial persona roleplay** — Practice Mode opponent
2. **Debrief generation** — Post-round feedback
3. **Pre-mortem scenario construction** — Worst-case preparation
4. **Cultural calibration** — Context-aware simulation tuning

All outputs are structured JSON. Each prompt is tested, output validated, and quality scored.

In [ ]:
# ── 0. Imports (model already loaded from Step 1) ────────────────────────────
import json
import re
from dataclasses import dataclass, field
from typing import Any

# ── Shared helpers ────────────────────────────────────────────────────────────

def chat(messages: list[dict], max_new_tokens: int = 512) -> str:
    """Thin wrapper around model.generate — identical to Step 1."""
    import torch
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


def parse_json(raw: str) -> dict | list:
    """Strip markdown fences and parse JSON. Raises ValueError on failure."""
    clean = re.sub(r"```(?:json)?|```", "", raw).strip()
    try:
        return json.loads(clean)
    except json.JSONDecodeError as e:
        raise ValueError(f"JSON parse failed: {e}\nRaw output:\n{raw}")


@dataclass
class PromptResult:
    name: str
    raw: str
    parsed: Any = None
    checks: dict = field(default_factory=dict)

    @property
    def passed(self) -> bool:
        return all(self.checks.values())


RESULTS: list[PromptResult] = []
print("Helpers ready")

In [ ]:
# ── 1. ADVERSARIAL PERSONA ROLEPLAY prompt ───────────────────────────────────
# Output schema:
# {
#   "persona_name": str,
#   "reply": str,          -- what the person says back
#   "resistance_type": str -- one of: deflect | guilt_trip | deny | counter_attack | silent | concede
#   "escalation_level": float  -- 0.0 (calm) to 1.0 (explosive)
# }

PERSONA_SYSTEM = """\
You are roleplaying as {persona_name}, a real person the user needs to have a difficult conversation with.

Behavioral profile:
{profile}

Conversation goal (from user's perspective): {user_goal}

Rules:
- Stay IN character. Never break the fourth wall.
- Be REALISTIC, not cooperative. This person has their own interests, defenses, and triggers.
- Respond to the user's last message exactly as {persona_name} would — use their vocabulary, tone, and patterns.

After your in-character reply, output ONLY this JSON block (no other text):
```json
{{
  "persona_name": "{persona_name}",
  "reply": "<your in-character reply>",
  "resistance_type": "<deflect|guilt_trip|deny|counter_attack|silent|concede>",
  "escalation_level": <0.0 to 1.0>
}}
```
"""

# Test scenario: salary negotiation — manager who deflects
persona_system = PERSONA_SYSTEM.format(
    persona_name="David (manager)",
    profile=(
        "Deflects with budget language when cornered. "
        "Hates being surprised in conversations. "
        "Uses 'we' language to dilute personal accountability. "
        "Responds better to data than emotion."
    ),
    user_goal="Ask for a 15% salary raise with market data as evidence.",
)

user_opening = (
    "David, I've been researching market rates and I've found that someone "
    "with my experience and skills earns about 15% more at comparable companies. "
    "I'd like to discuss adjusting my salary to match that."
)

raw_persona = chat([
    {"role": "system", "content": persona_system},
    {"role": "user",   "content": user_opening},
], max_new_tokens=384)

print("Raw output:\n", raw_persona)
print()

In [ ]:
# Validate persona output
parsed_persona = parse_json(raw_persona)

VALID_RESISTANCE = {"deflect", "guilt_trip", "deny", "counter_attack", "silent", "concede"}
persona_result = PromptResult(
    name="adversarial_persona",
    raw=raw_persona,
    parsed=parsed_persona,
    checks={
        "has_reply":           bool(parsed_persona.get("reply")),
        "valid_resistance":    parsed_persona.get("resistance_type") in VALID_RESISTANCE,
        "escalation_in_range": 0.0 <= float(parsed_persona.get("escalation_level", -1)) <= 1.0,
        "persona_name_present": bool(parsed_persona.get("persona_name")),
    }
)
RESULTS.append(persona_result)

print("Parsed output:")
print(json.dumps(parsed_persona, indent=2))
print()
for check, ok in persona_result.checks.items():
    print(f"  {'✅' if ok else '❌'}  {check}")
print(f"\nAdversarial Persona: {'PASS ✅' if persona_result.passed else 'FAIL ❌'}")

In [ ]:
# ── 2. DEBRIEF GENERATION prompt ─────────────────────────────────────────────
# Output schema:
# {
#   "ground_lost": [{"turn": int, "quote": str, "reason": str}],
#   "over_apologies": [{"turn": int, "quote": str}],
#   "missed_openings": [{"turn": int, "description": str, "better_line": str}],
#   "wins": [{"turn": int, "description": str}],
#   "one_fix_next_time": str
# }

DEBRIEF_SYSTEM = """\
You are a conversation coach analyzing a practice round for Tough Talks.
The user was trying to: {user_goal}

Analyze the transcript below and produce a debrief in this EXACT JSON format:
{{
  "ground_lost": [
    {{"turn": <int>, "quote": "<exact quote>", "reason": "<why this lost ground>"}}
  ],
  "over_apologies": [
    {{"turn": <int>, "quote": "<exact quote>"}}
  ],
  "missed_openings": [
    {{"turn": <int>, "description": "<what was missed>", "better_line": "<the line that would have worked>"}}
  ],
  "wins": [
    {{"turn": <int>, "description": "<what went well and why>"}}
  ],
  "one_fix_next_time": "<single most important thing to do differently>"
}}

Rules:
- Only output the JSON block. No preamble, no markdown, no explanation.
- "turn" is the 1-based index of the user's turns only.
- "better_line" must be a complete, ready-to-say sentence the user could use verbatim.
- Be specific and honest. Vague praise is useless.
"""

SAMPLE_PRACTICE_TRANSCRIPT = """
User (turn 1): David, I've been looking at market rates and I think there might be a gap...
David: We appreciate everything you do. Budget is tight right now for the whole team.
User (turn 2): Oh, I totally understand, I'm sorry to bring it up at a bad time — it's just something I've been thinking about.
David: We'll keep it in mind for the annual review cycle.
User (turn 3): Right, yeah, that makes sense. I just feel like maybe we could talk about it then? Sorry.
David: Sure, we'll loop back. Is there anything else?
User (turn 4): No, no, that's fine. Thank you for your time.
"""

debrief_system = DEBRIEF_SYSTEM.format(
    user_goal="Ask for a 15% raise with market data as evidence."
)

raw_debrief = chat([
    {"role": "system", "content": debrief_system},
    {"role": "user",   "content": f"Transcript:\n{SAMPLE_PRACTICE_TRANSCRIPT}"},
], max_new_tokens=600)

print("Raw output:\n", raw_debrief)
print()

In [ ]:
# Validate debrief output
parsed_debrief = parse_json(raw_debrief)

REQUIRED_KEYS = {"ground_lost", "over_apologies", "missed_openings", "wins", "one_fix_next_time"}
debrief_result = PromptResult(
    name="debrief_generation",
    raw=raw_debrief,
    parsed=parsed_debrief,
    checks={
        "all_keys_present":      REQUIRED_KEYS.issubset(set(parsed_debrief.keys())),
        "ground_lost_is_list":   isinstance(parsed_debrief.get("ground_lost"), list),
        "missed_openings_has_better_line": all(
            "better_line" in item
            for item in parsed_debrief.get("missed_openings", [])
        ),
        "one_fix_is_string":     isinstance(parsed_debrief.get("one_fix_next_time"), str),
        "one_fix_non_empty":     bool(parsed_debrief.get("one_fix_next_time", "").strip()),
    }
)
RESULTS.append(debrief_result)

print("Parsed output:")
print(json.dumps(parsed_debrief, indent=2))
print()
for check, ok in debrief_result.checks.items():
    print(f"  {'✅' if ok else '❌'}  {check}")
print(f"\nDebrief Generation: {'PASS ✅' if debrief_result.passed else 'FAIL ❌'}")

In [ ]:
# ── 3. PRE-MORTEM SCENARIO CONSTRUCTION prompt ───────────────────────────────
# Output schema:
# {
#   "goal": str,
#   "failure_scenarios": [
#     {
#       "scenario_id": int,
#       "title": str,
#       "description": str,
#       "likely_trigger": str,
#       "destabilization_risk": float,   -- 0.0 to 1.0
#       "simulation_parameters": {
#           "resistance_type": str,
#           "escalation_ceiling": float,
#           "opening_move": str
#       }
#     }
#   ]
# }

PREMORTEM_SYSTEM = """\
You are a strategic conversation coach preparing a user for a difficult talk.

Your task: generate the 3 most realistic failure scenarios for the conversation described.
Each scenario is a specific way the conversation could go badly wrong.

Output ONLY this JSON — no preamble, no markdown:
{{
  "goal": "<user's stated goal>",
  "failure_scenarios": [
    {{
      "scenario_id": 1,
      "title": "<short name for this failure>",
      "description": "<how this scenario plays out in 2-3 sentences>",
      "likely_trigger": "<what user says or does that triggers this>",
      "destabilization_risk": <0.0 to 1.0>,
      "simulation_parameters": {{
        "resistance_type": "<deflect|guilt_trip|deny|counter_attack|silent|concede>",
        "escalation_ceiling": <0.0 to 1.0>,
        "opening_move": "<the opponent's first line in this scenario>"
      }}
    }}
  ]
}}
"""

raw_premortem = chat([
    {"role": "system", "content": PREMORTEM_SYSTEM},
    {"role": "user",   "content": (
        "I need to tell my co-founder that I think we should shut down the startup. "
        "He has invested 3 years and his savings. I am the CEO. "
        "We have a board meeting in 2 weeks. "
        "My goal is to reach a mutual decision to wind down gracefully, "
        "preserve the friendship, and agree on the shutdown timeline."
    )},
], max_new_tokens=700)

print("Raw output:\n", raw_premortem)
print()

In [ ]:
# Validate pre-mortem output
parsed_premortem = parse_json(raw_premortem)

scenarios = parsed_premortem.get("failure_scenarios", [])
premortem_result = PromptResult(
    name="premortem_generation",
    raw=raw_premortem,
    parsed=parsed_premortem,
    checks={
        "has_goal":               bool(parsed_premortem.get("goal")),
        "exactly_3_scenarios":    len(scenarios) == 3,
        "all_have_sim_params":    all("simulation_parameters" in s for s in scenarios),
        "all_have_opening_move":  all(
            s.get("simulation_parameters", {}).get("opening_move") for s in scenarios
        ),
        "destab_risk_valid":      all(
            0.0 <= float(s.get("destabilization_risk", -1)) <= 1.0 for s in scenarios
        ),
    }
)
RESULTS.append(premortem_result)

print("Parsed output:")
print(json.dumps(parsed_premortem, indent=2))
print()
for check, ok in premortem_result.checks.items():
    print(f"  {'✅' if ok else '❌'}  {check}")
print(f"\nPre-Mortem Generation: {'PASS ✅' if premortem_result.passed else 'FAIL ❌'}")

In [ ]:
# ── 4. CULTURAL CALIBRATION prompt ───────────────────────────────────────────
# Output schema:
# {
#   "context": {"culture": str, "industry": str, "power_dynamic": str},
#   "communication_adjustments": {
#       "directness_level": float,        -- 0.0 indirect to 1.0 very direct
#       "silence_norm": str,              -- description of how silence is used
#       "face_saving_required": bool,
#       "typical_refusal_style": str
#   },
#   "simulation_instructions": str,       -- what the persona sim should do differently
#   "opening_line_recommendations": [str] -- 3 culturally calibrated opening options
# }

CULTURAL_SYSTEM = """\
You are a cross-cultural communication specialist for Tough Talks.

Given a conversation context, produce cultural calibration parameters so the simulation 
accurately reflects how communication actually works in that context.

Output ONLY this JSON — no preamble, no markdown:
{{
  "context": {{
    "culture": "<culture or country>",
    "industry": "<industry or domain>",
    "power_dynamic": "<e.g. employee-to-manager, peer-to-peer, child-to-parent>"
  }},
  "communication_adjustments": {{
    "directness_level": <0.0 to 1.0>,
    "silence_norm": "<how silence is used and interpreted in this context>",
    "face_saving_required": <true|false>,
    "typical_refusal_style": "<how people say no or deflect in this context>"
  }},
  "simulation_instructions": "<2-3 sentences instructing the persona sim how to behave>",
  "opening_line_recommendations": [
    "<option 1>",
    "<option 2>",
    "<option 3>"
  ]
}}
"""

raw_cultural = chat([
    {"role": "system", "content": CULTURAL_SYSTEM},
    {"role": "user",   "content": (
        "Context: I am a junior engineer at a Japanese tech company. "
        "I need to tell my senior manager (senpai) that I disagree with a technical "
        "decision he has publicly committed to. "
        "Industry: software. Power dynamic: junior-to-senior, hierarchical culture."
    )},
], max_new_tokens=500)

print("Raw output:\n", raw_cultural)
print()

In [ ]:
# Validate cultural calibration output
parsed_cultural = parse_json(raw_cultural)

adj = parsed_cultural.get("communication_adjustments", {})
cultural_result = PromptResult(
    name="cultural_calibration",
    raw=raw_cultural,
    parsed=parsed_cultural,
    checks={
        "has_context":             bool(parsed_cultural.get("context")),
        "directness_in_range":     0.0 <= float(adj.get("directness_level", -1)) <= 1.0,
        "face_saving_is_bool":     isinstance(adj.get("face_saving_required"), bool),
        "has_sim_instructions":    bool(parsed_cultural.get("simulation_instructions")),
        "has_3_opening_lines":     len(parsed_cultural.get("opening_line_recommendations", [])) == 3,
    }
)
RESULTS.append(cultural_result)

print("Parsed output:")
print(json.dumps(parsed_cultural, indent=2))
print()
for check, ok in cultural_result.checks.items():
    print(f"  {'✅' if ok else '❌'}  {check}")
print(f"\nCultural Calibration: {'PASS ✅' if cultural_result.passed else 'FAIL ❌'}")

In [ ]:
# ── 5. Serialize prompt registry to disk ─────────────────────────────────────
# Store all validated prompts as a reusable registry for later steps.
import pathlib

PROMPT_REGISTRY = {
    "adversarial_persona": {
        "system_template": PERSONA_SYSTEM,
        "required_inputs": ["persona_name", "profile", "user_goal"],
        "output_schema": {
            "persona_name": "str",
            "reply": "str",
            "resistance_type": "deflect|guilt_trip|deny|counter_attack|silent|concede",
            "escalation_level": "float 0.0-1.0"
        }
    },
    "debrief_generation": {
        "system_template": DEBRIEF_SYSTEM.replace("\n", "\\n"),   # safe for JSON
        "required_inputs": ["user_goal", "transcript"],
        "output_schema": {
            "ground_lost": "list[{turn, quote, reason}]",
            "over_apologies": "list[{turn, quote}]",
            "missed_openings": "list[{turn, description, better_line}]",
            "wins": "list[{turn, description}]",
            "one_fix_next_time": "str"
        }
    },
    "premortem_generation": {
        "system_template": PREMORTEM_SYSTEM,
        "required_inputs": ["conversation_description"],
        "output_schema": {
            "goal": "str",
            "failure_scenarios": "list[{scenario_id, title, description, likely_trigger, destabilization_risk, simulation_parameters}]"
        }
    },
    "cultural_calibration": {
        "system_template": CULTURAL_SYSTEM,
        "required_inputs": ["culture", "industry", "power_dynamic", "conversation_goal"],
        "output_schema": {
            "context": "dict",
            "communication_adjustments": "dict",
            "simulation_instructions": "str",
            "opening_line_recommendations": "list[str, str, str]"
        }
    }
}

out_path = pathlib.Path("tough_talks_prompt_registry.json")
out_path.write_text(json.dumps(PROMPT_REGISTRY, indent=2))
print(f"Prompt registry saved → {out_path.resolve()}")

In [ ]:
# ── 6. Step 2 final summary ───────────────────────────────────────────────────
print("═" * 55)
print("STEP 2 RESULTS — Prompt Engineering")
print("═" * 55)

all_pass = True
for r in RESULTS:
    status = "PASS ✅" if r.passed else "FAIL ❌"
    print(f"{status}  {r.name}")
    if not r.passed:
        all_pass = False
        for check, ok in r.checks.items():
            if not ok:
                print(f"         └─ ❌ {check}")

print()
print("Prompt registry:", "saved ✅" if pathlib.Path("tough_talks_prompt_registry.json").exists() else "missing ❌")
print()
print("PHASE 1 STATUS:", "✅ COMPLETE — READY FOR PHASE 2" if all_pass else "⚠️  FIX FAILURES THEN PROCEED")
print()
if not all_pass:
    print("Next actions:")
    print("  1. Review raw outputs for failed prompts above")
    print("  2. Adjust the system prompt for the failing feature")
    print("  3. If E2B consistently fails a prompt, flag it for 27B comparison")
    print("  4. Report back with results before proceeding to Phase 2")